# M. A. Halim — Toddler training (free Google Colab)

Trains Halim's first language brain on **your** 2,580 trading pairs.

**Before opening this notebook:** on your Mac run `./scripts/halim_package_colab.sh` and upload `halim_sft.zip` in Step 4.

In [ ]:
# Step 1 — Confirm GPU (must say Tesla T4 or similar)
!nvidia-smi

In [ ]:
# Step 2 — Install training libraries (~2 min)
!pip install -q "transformers>=4.44" "datasets>=2.20" "peft>=0.12" "trl>=0.9" "accelerate>=0.33" "bitsandbytes>=0.43"

In [ ]:
# Step 3 — Upload halim_sft.zip from your Mac
from google.colab import files
uploaded = files.upload()  # pick halim_sft.zip
print('Uploaded:', list(uploaded.keys()))

In [ ]:
# Step 4 — Delete OLD script, unzip fresh zip (important!)
import os
from pathlib import Path

for f in ("train_toddler_colab.py", "toddler_v1"):
    p = Path(f)
    if p.is_file():
        p.unlink()
    elif p.is_dir():
        import shutil
        shutil.rmtree(p)

!unzip -o halim_sft.zip -d .
!ls -la sft/ train_toddler_colab.py

In [ ]:
# Step 5 — VERIFY script version (must say v2 / _build_sft_config)
from pathlib import Path

script = Path("train_toddler_colab.py")
text = script.read_text()
if "_build_sft_config" not in text or "training_args = SFTConfig(" in text:
    raise RuntimeError(
        "WRONG SCRIPT in Colab — still the old broken file.\n"
        "Fix: Runtime → Restart session, re-upload halim_sft.zip from Mac, re-run cells 1–5.\n"
        "Mac path: ~/Downloads/tradingbot/halim_sft.zip"
    )
print("✅ Script OK (halim-toddler-v2)")
print("Line 135:", [l.strip() for l in text.splitlines()[134:136]])

In [ ]:
# Step 6 — TRAIN Halim (~15–30 min)
!python train_toddler_colab.py

In [ ]:
# Step 7 — Quick test (tokenizer from base model + weights from merged/)
from transformers import AutoModelForCausalLM, AutoTokenizer
import torch
from pathlib import Path

BASE = "Qwen/Qwen2.5-0.5B-Instruct"
path = "toddler_v1/merged"

if not Path(path).is_dir():
    raise FileNotFoundError(f"Missing {path} — wait for training + merge to finish")

tok = AutoTokenizer.from_pretrained(BASE, trust_remote_code=True)
model = AutoModelForCausalLM.from_pretrained(
    path, torch_dtype=torch.float16, device_map="auto", trust_remote_code=True,
)
msgs = [{"role": "user", "content": "Ticker SOFI score 92 vol 1.5x — enter or skip?"}]
prompt = tok.apply_chat_template(msgs, tokenize=False, add_generation_prompt=True)
inputs = tok(prompt, return_tensors="pt").to(model.device)
out = model.generate(**inputs, max_new_tokens=120, pad_token_id=tok.eos_token_id)
print(tok.decode(out[0], skip_special_tokens=True))

In [ ]:
# Step 8 — Download toddler_v1 folder to your Mac (~1GB)
!zip -r halim_toddler_v1.zip toddler_v1/
from google.colab import files
files.download('halim_toddler_v1.zip')